In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Concise Implementation

In that section we wrote every part of linear regression
by hand, from the initialization of the parameters to the update of
minibatch stochastic gradient descent. In that section we
arranged the same parts into classes and trained the model once more
through `d2l.Trainer`. Both steps were feasible because the model has three
parameters. The models of later chapters are far larger. The image
classifier ResNet-50 of that section, for instance, has 25.6
million parameters in 161 tensors, and it is trained with update rules that
keep a running average of past gradients for every parameter. Writing and
testing all of this by hand for every new model would take most of our
effort. Worse, a mistake in any part rarely raises an error. The model
still trains, only less well.

Deep learning frameworks such as MXNet [@Chen.Li.Li.ea.2015], JAX
[@Frostig.Johnson.Leary.2018] with its libraries Flax and Optax,
PyTorch [@Paszke.Gross.Massa.ea.2019], and TensorFlow
[@Abadi.Barham.Chen.ea.2016] therefore provide these parts ready-made.
Their layers, losses, and optimizers are tested by many users and run on
code tuned for each kind of hardware, and one loss or optimizer can be
swapped for another by changing a line. This section swaps our hand-written
parts for the framework's. A fully connected layer holds the weights and
the bias and computes the predictions, a built-in loss computes the squared
error, and an optimizer object applies the update. The `Trainer` runs
unchanged, because it calls only the methods that every `Module` has. Only
the scale of the loss needs some care.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## The Fully Connected Layer

Linear regression is a single fully connected layer
(the figure), and every framework provides that layer.
For a minibatch with design matrix $\mathbf{X}$ it computes the predictions
$\mathbf{X}\mathbf{w} + b$ that the `forward` method of our scratch model
wrote out, and it creates and stores its own parameters. All that is left
for us to choose is the number of outputs, here 1, and how the parameters
are initialized. As in the scratch model, the weights are drawn from a
normal distribution with mean 0 and standard deviation 0.01, and the bias
starts at 0.

PyTorch's `nn.LazyLinear(1)` takes the number $d$ of inputs from the first
input it receives and has no parameters until then, which spares us
counting the inputs of every layer in deeper networks. The method
`weight_init` states our initialization for any linear layer; from
that section on, the trainer applies it after the first forward
pass.

In [ ]:
class LinearRegression(d2l.Module):
    """Linear regression built from framework components."""
    def __init__(self, lr):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.LazyLinear(1)

    def weight_init(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.01)
            nn.init.zeros_(module.bias)

The model needs no `forward` method of its own. It inherits the `forward`
of `Module` (that section), which passes its input to
`self.net`, here the layer.

## The Loss Function

Our scratch loss was half the squared error, averaged over the minibatch
$\mathcal{B}$. Most frameworks provide the *mean squared error* instead,

$$\frac{1}{|\mathcal{B}|}\sum_{i \in \mathcal{B}} \left(\hat{y}^{(i)} - y^{(i)}\right)^2,$$

which has no factor $\frac{1}{2}$. It is the usual measure of prediction
quality in regression: its square root is the typical size of an error, in
the units of the label. The two losses have the same minimizer, so what we
fit does not change. The steps do. A learning rate has a meaning only
together with the scale of its loss
(that section), and the framework loss
is twice ours. To repeat the steps that the scratch model took at
$\eta = 0.03$, this model must therefore be trained at $\eta = 0.015$.

The loss also decides what becomes of the silent error of
that section, in which labels of shape $(32,)$ are broadcast
against predictions of shape $(32, 1)$ into a $32 \times 32$ matrix of
differences.

`nn.MSELoss` averages the squared errors over all entries of its inputs,
here the examples of the minibatch. If labels and predictions differ in
shape, it warns and then averages over the $32 \times 32$ matrix of
differences anyway, so training proceeds on the wrong loss.

In [ ]:
@d2l.add_to_class(LinearRegression)
def loss(self, y_hat, y):
    fn = nn.MSELoss()
    return fn(y_hat, y)

## The Optimizer

The optimizer replaces our `SGD` class of that section. It
receives the parameters and the learning rate $\eta$, and at each step it subtracts $\eta$ times the
gradient of the minibatch loss from each parameter, the update we wrote by
hand. The optimizer finds the parameters through the layers of the model,
so a model with more layers needs no change here. Each framework offers the
other common update rules behind the same interface, and replacing minibatch
SGD by one of them changes only this one line.

`torch.optim.SGD` takes the parameters as `self.parameters()`, which
collects the parameters of every layer of the model, here the weight and
the bias of `net`.

In [ ]:
@d2l.add_to_class(LinearRegression)
def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), self.lr)

## Training

We train on data drawn as before, from the generator of
that section with $\mathbf{w}^* = (2, -3.4)$,
$b^* = 4.2$, and noise of standard deviation $\sigma = 0.01$: 1,000
training examples, served in minibatches of 32, and 1,000 validation
examples. From the same initial parameters and on the same minibatches,
the two models would compute the same updates (an exercise arranges this).
Here the examples, their order, and the initial weights are all drawn
afresh, so we should expect the behavior of the scratch run but not its
digits.

Before training we call the model once on one example to create the
parameters of the lazy layer, and `apply` then calls `weight_init` on every
layer of `net`. Without this line the layer would keep PyTorch's default
initialization, a uniform distribution whose width shrinks with the number
of inputs.

We also put the trainer's plot, `trainer.board`, on a logarithmic scale, so
that the loss stays visible after it has fallen by several orders of
magnitude.

In [ ]:
data = d2l.SyntheticRegressionData(w=d2l.tensor([2, -3.4]), b=4.2)
model = LinearRegression(lr=0.015)
model(data.X[:1])  # Creates the parameters of the lazy layer
model.net.apply(model.weight_init)
trainer = d2l.Trainer(max_epochs=10)
trainer.board.yscale = 'log'
trainer.fit(model, data)

The plot shows the mean training loss of each epoch and the validation loss
after each epoch. Both curves fall along a straight line, by a factor of
about 7 per epoch, and level off near $10^{-4}$ after about eight epochs.
We have seen this plot before. The slope is the one that
that section predicted and measured, as it must be if the
steps are the same. Only the height differs, since every loss is now
doubled. In particular the floor, the loss of the true parameters, is the
noise variance $\sigma^2 = 10^{-4}$ and no longer $\sigma^2/2$. To compare
the run with this number we read the final validation loss from
`trainer.history`.

In [ ]:
print(f"validation loss: {trainer.history['val_loss'][-1]:.2e}")

The final validation loss lies within about 15% of $\sigma^2 = 10^{-4}$.

What about the parameters themselves? The scratch model kept them as
`self.w` and `self.b`. Here they belong to the layer, and `get_w_b` reads
them out of `net`. A layer stores its weight as a matrix (a $1 \times d$ row
in Gluon and PyTorch, a $d \times 1$ column called the `kernel` in Keras and
Flax), so the comparison below first reshapes it to the shape of
$\mathbf{w}^*$.

In [ ]:
@d2l.add_to_class(LinearRegression)
def get_w_b(self):
    return self.net.weight.detach(), self.net.bias.detach()

In [ ]:
w, b = model.get_w_b()
w_error = d2l.numpy(d2l.reshape(w, data.w.shape) - data.w)
b_error = d2l.numpy(b - data.b)
print('error in w:', ', '.join(f'{e:.1e}' for e in w_error))
print('error in b:', ', '.join(f'{e:.1e}' for e in b_error))

The parameter errors $\hat{\mathbf{w}} - \mathbf{w}^*$ and $\hat{b} - b^*$
are typically a few times $10^{-4}$ and rarely exceed $10^{-3}$. The
scratch model left errors of the same size, and they have the same sources
(that section), chiefly the noise in the 1,000 training
labels and what ten epochs leave of the initial error. Neither depends on
who wrote the layer.

The rest of the book builds its models this way, from the layers, losses,
and optimizers of the framework, and writes a part by hand only to show how
it works or when the framework does not provide it. One habit of this
chapter cannot be kept, however: judging a trained model by its distance
from parameters that we chose ourselves. With real data there are no true
parameters to compare with, and what matters is how well a model predicts
examples it has not seen, the question of
that section.

## Summary

Deep learning frameworks provide tested and fast versions of the parts we
wrote by hand for linear regression. A fully connected layer holds the
weights and the bias and computes the predictions, a loss function computes
the mean squared error, and an optimizer applies the SGD update to every
parameter of the model. Replacing each hand-written part by its framework
counterpart leaves `Module`, `DataModule`, and `Trainer` unchanged. A
framework loss may differ from ours by a constant factor, and the learning
rate must then be rescaled to match. Writing a part by hand remains the way
to understand it, and the way to build one that the framework lacks.

## Exercises

1. [code] **Loss scaling.** The framework loss in `LinearRegression.loss`
   is twice the loss of `LinearRegressionScratch`.
    1. For one minibatch $\mathcal{B}$ of `SyntheticRegressionData`,
       compute the gradient of `LinearRegression.loss` with respect to the
       weights by automatic differentiation, and compare it entry by entry
       with the formula
       $\frac{2}{|\mathcal{B}|}\sum_{i \in \mathcal{B}} \mathbf{x}^{(i)}\left(\mathbf{w}^\top \mathbf{x}^{(i)} + b - y^{(i)}\right)$
       at the same parameters.
    1. Give `LinearRegression` and `LinearRegressionScratch` the same
       initial parameters and the same minibatches, for example by setting
       `batch_size` to `num_train` so that every step uses the whole
       training set. Train the first at `lr=0.015` and the second at
       `lr=0.03`, confirm that the trained parameters agree, and explain
       why the two training-loss curves still differ.

1. [code] **A layer without a bias.** Build `LinearRegression` with a
   layer that has no bias (every framework's dense layer has an argument
   that turns the bias off), and train it as in this section.
    1. Before training, predict the value at which the validation loss
       levels off, from the expected loss
       the equation doubled for the mean
       squared error. Then check your prediction.
    1. Predict what the weights converge to and how far they end from
       $\mathbf{w}^*$. Since the features have mean zero, the missing bias
       acts on the weights like label noise of standard deviation
       $b^* = 4.2$; it enters both the estimation error of
       that section and the noise
       ball of minibatch SGD (that section). Compare
       your prediction with the errors of the trained weights.

1. [code] **Several outputs.** Change the output width of the layer in
   `LinearRegression` from 1 to 2. After constructing `data`, append to
   `data.y` a second label column
   $y_2 = \mathbf{x}^\top \mathbf{w}_2^* + b_2^* + \epsilon_2$, computed
   from `data.X` with generating parameters of your choice and noise of
   the same size.
    1. Show that the squared loss summed over the two outputs splits into
       two independent linear regressions, one per output. Conclude that
       the parameters learned for one output do not depend on the labels of
       the other.
    1. The framework losses average over every entry of the
       $|\mathcal{B}| \times 2$ matrix of errors of a minibatch
       $\mathcal{B}$, not only over its examples. State by which factor
       the gradient of each output's parameters shrinks relative to the
       single-output model, and which learning rate reproduces the
       single-output training run. Confirm this by training both models
       from the same initial parameters on the same minibatches.

1. [code] **Feature scale and step size.** Multiply the first column of
   `data.X` by 100 after constructing `data`, leaving `data.y` unchanged,
   so that the generating weight of that feature becomes $0.02$. Recall
   from exercise 4 that
   gradient descent diverges once $\eta$ exceeds $2/\mu_{\max}$, where
   $\mu_{\max}$ is the largest eigenvalue of the Hessian $\mathbf{H}$ of
   the loss.
    1. Compute $\mathbf{H}$ for `LinearRegression.loss` and estimate its
       eigenvalues before and after the rescaling.
    1. Predict the largest learning rate for which training does not
       diverge, before and after the rescaling, and check both predictions
       by training at learning rates around them.
    1. At a learning rate just below the new limit, predict how many
       updates the second weight needs to reduce its error by a factor of
       100, and compare the prediction with a measurement. Then standardize
       the rescaled feature as in that section, and show that
       `lr=0.015` works again.

1. [code] **Optimization versus estimation error.** ● Train
   `LinearRegression` with `lr=0.015` and `max_epochs=10` on
   `SyntheticRegressionData` with `num_train` set to each of
   $5, 50, 500, 5{,}000$ and `batch_size=min(32, num_train)`. For each run,
   compare the distance $\|\hat{\mathbf{w}} - \mathbf{w}^*\|_2$ of the
   trained weights from the generating weights $\mathbf{w}^*$, stored in
   `data.w`, with the same distance for the least-squares solution on the
   same training data, computed as in the recovery subsection
   (that section).
    1. Count the parameter updates in each run, and explain the ratio of
       the two errors at each `num_train`.
    1. Find the number of epochs after which the run with `num_train=5`
       comes within a factor of two of its least-squares error.

[Discussions](https://d2l.discourse.group/t/45)